# Adversarial TTS - Class-Based Architecture

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

## 1. Imports

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

In [1]:
import torch
import os
import argparse
import matplotlib.pyplot as plt

%cd StyleTTS2

from Datastructures.enum import AttackMode, FitnessObjective
from Datastructures.dataclass import ModelData

# Import class-based modules
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger
from Trainer.GraphPlotter import GraphPlotter
from Trainer.VectorManipulator import VectorManipulator

# Import Pymoo components
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

# Import ObjectiveManager
from Objectives.ObjectiveManager import ObjectiveManager

from helper import write_run_summary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

Device: cpu
Available GPUs: 0


## 2. Configuration

Edit the values below to configure the optimization run.

In [2]:
# =============================================================================
# Configuration - Edit these values directly
# =============================================================================
args = argparse.Namespace(
    # Text parameters
    ground_truth_text="I think the NFL is lame and boring",
    target_text="The Seattle Seahawks are the best Team in the world",

    # Optimization parameters
    loop_count=1,
    num_generations=1,
    pop_size=1,
    iv_scalar=0.5,
    size_per_phoneme=1,
    batch_size=100,  # -1 for full batch

    # Flags
    notify=False,
    subspace_optimization=False,

    # Mode and objectives
    mode="TARGETED",  # TARGETED, UNTARGETED, NOISE_UNTARGETED
    ACTIVE_OBJECTIVES=["PESQ", "WER_GT"],
    thresholds=["PESQ=0.3", "WER_GT=0.5"],
)

## 3. Initialize Environment

Load models and prepare audio data.

In [3]:
# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(device)
config_data = loader.load_configuration(args)
config_data.print_summary()

tts_model, asr_model = loader.load_required_models()

audio_gt, audio_target, audio_embedding_gt, audio_embedding_target = loader.generate_audio_data(
    config_data.mode, config_data.text_gt, config_data.text_target, tts_model
)

print("\nEnvironment initialized successfully!")

=== Configuration ===
GT Text:               I think the NFL is lame and boring
Target Text:           The Seattle Seahawks are the best Team in the world
Generations:           1
Population Size:       1
Loop Count:            1
IV Scalar:             0.5
Size Per Phoneme:      1
Batch Size:            1
Subspace Optimization: False
Notify (WhatsApp):     False
Mode:                  TARGETED
Objectives:            ['PESQ', 'WER_GT']
Thresholds:            PESQ<=0.3, WER_GT<=0.5
Loading TTS Model (StyleTTS2)...
Loading ASR Model (Whisper)...

Environment initialized successfully!


## 4. Setup Trainer Components

Initialize ObjectiveManager, VectorManipulator, Trainer, and Logger.

In [4]:
# Initialize Objective Manager
objective_manager = ObjectiveManager(
    model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
    active_objectives=config_data.active_objectives,
    device=device,
    text_gt=config_data.text_gt,
    text_target=config_data.text_target,
    mode=config_data.mode,
    audio_gt=audio_gt,
    style_vector_acoustic=audio_embedding_gt.style_vector_acoustic,
    style_vector_prosodic=audio_embedding_gt.style_vector_prosodic
)

vector_manipulator = VectorManipulator(audio_embedding_gt, audio_embedding_target.h_text, config_data)

# Create Trainer and Logger
trainer = AdversarialTrainer(
    tts_model, asr_model, config_data.active_objectives, 
    config_data.thresholds, objective_manager, vector_manipulator, device
)
logger = RunLogger(
    config_data.active_objectives, tts_model, asr_model, vector_manipulator, device
)

print("Trainer components initialized!")

Initialized PESQ (batching=False)
Initialized WER_GT (batching=True)
Trainer components initialized!


## 5. Run Optimization

Run the optimization loop and log results.

In [5]:
for loop_iteration in range(config_data.loop_count):
    print(f"\n[Loop {loop_iteration + 1}/{config_data.loop_count}] Starting optimization loop...")

    # Initialize fresh optimizer for this cycle
    optimizer = PymooOptimizer(
        bounds=(0, 1),
        algorithm=NSGA2,
        algo_params={"pop_size": config_data.pop_size},
        num_objectives=len(config_data.active_objectives),
        solution_shape=(audio_embedding_gt.input_length.detach().cpu().item(), config_data.size_per_phoneme),
    )

    fitness_data, generation_count, elapsed_time_total = trainer.run_full_iteration(optimizer, config_data.num_generations, config_data.pop_size, config_data.batch_size)

    # 5. Log Results
    folder_path = logger.setup_output_directory()
    logger.save_fitness_history(fitness_data)

    best_candidate = logger.select_best_candidate(optimizer.best_candidates, config_data.thresholds)
    audio_best, text_best, audio_embedding_best = logger.run_final_inference(best_candidate)

    logger.save_audios(audio_gt, audio_target, audio_best)
    logger.save_torch_state(text_best, audio_embedding_best, best_candidate, config_data)

    # 6. Generate Graphs
    graph_plotter = GraphPlotter(config_data.active_objectives, generation_count, folder_path, fitness_data)
    graph_plotter.generate_hypervolume_graph()
    graph_plotter.generate_pareto_population_graph()
    graph_plotter.generate_mean_population_graph()
    graph_plotter.generate_minimal_population_graph()
    plt.close('all')

    # 7. Write Summary to Terminal
    write_run_summary(folder_path, text_best, best_candidate, generation_count, elapsed_time_total, config_data)

    print("[Log] Finished saving all results")


[Loop 1/1] Starting optimization loop...
Press Ctrl+C to stop training early and save results.


Generations:   0%|          | 0/1 [00:00<?, ?it/s]

[Gen 1] PESQ: 0.5420 (Avg: 0.5420) | WER_GT: 1.0000 (Avg: 1.0000)
[Log] Output directory initialized: outputs/PESQ_WER_GT/20260116_1722
[Log] Full fitness history saved as fitness_history.csv

[Log] Candidates on Pareto Front: 1
[Log] No candidate met thresholds. Preceding with all candidates.
[Log] Selected Candidate Fitness: [0.5419512987136841, 1.0]


/home/yanis/miniconda3/envs/styletts2/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[Log] Torch state saved as reconstruction_pack.pt
[Log] Mean fitness graph saved as mean_fitness_stack.png
[Log] Minimal fitness graph saved as minimal_fitness_stack.png
[Log] Finished saving all results
